#Part 0: Mounting to google drive for using colab

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

import sys, os
sys.path.insert(0, "/content/drive/MyDrive/cs209b/")

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

# ── Configure all paths here ──
ROOT      = Path("/content/drive/MyDrive/cs209b/")
ORIG_DIR  = ROOT / "MS2" / "original_data"   # raw CSVs live here
DATA_DIR  = ROOT / "MS2" / "data"             # outputs go here
OHLCV_DIR = ORIG_DIR / "local_ohlcv_10y_full" # per-ticker OHLCV cache

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OHLCV_DIR, exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"ORIG_DIR  : {ORIG_DIR}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OHLCV_DIR : {OHLCV_DIR}")

ROOT      : /content/drive/MyDrive/cs209b
ORIG_DIR  : /content/drive/MyDrive/cs209b/EDA/original_data
DATA_DIR  : /content/drive/MyDrive/cs209b/EDA/data
OHLCV_DIR : /content/drive/MyDrive/cs209b/EDA/original_data/local_ohlcv_10y_full


# Part 1: Imports

In [ ]:
import datetime as dt
import re
import time
from contextlib import redirect_stderr, redirect_stdout
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import yfinance.shared as yf_shared

# OHLCV field names used throughout
FIELDS = ["Open", "High", "Low", "Close", "Volume"]

# Part 2: Utility Fuctions

## 2.1: Make OHLCV columns

In [ ]:
def make_feature_columns() -> List[str]:
    """Generate the 55 feature-column names: t_minus_1..10 x 5 fields + t_plus_1 x 5 fields."""
    cols: List[str] = []
    for i in range(1, 11):
        for f in FIELDS:
            cols.append(f"t_minus_{i}_{f.lower()}")
    for f in FIELDS:
        cols.append(f"t_plus_1_{f.lower()}")
    return cols

## 2.2 Ticker normalization & helpers

In [ ]:
def normalize_query_ticker(ticker: str) -> str:
    """Normalize ticker for yfinance queries (e.g. BRK.B -> BRK-B)."""
    if not isinstance(ticker, str):
        return ""
    return ticker.strip().replace(".", "-").replace("/", "-")


def safe_filename(name: str) -> str:
    """Sanitize a string for use as a filename."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)

## 2.3 URL merging (processed <-> raw)

In [ ]:
def merge_url(processed: pd.DataFrame, raw: pd.DataFrame) -> pd.Series:
    """
    Merge URLs from the raw dataset onto the processed dataset.
    Primary match: same Unnamed: 0 index + title/headline + stock must agree.
    Fallback: stock + title/headline first non-null url.
    """
    raw_idx = raw[["Unnamed: 0", "headline", "stock", "url"]].rename(
        columns={"stock": "raw_stock", "url": "raw_url"}
    )
    m = processed[["Unnamed: 0", "title", "stock"]].merge(raw_idx, on="Unnamed: 0", how="left")
    valid_primary = (
        m["raw_url"].notna() & (m["stock"] == m["raw_stock"]) & (m["title"] == m["headline"])
    )
    out = m["raw_url"].where(valid_primary)

    lookup = raw[["stock", "headline", "url"]].dropna(subset=["url"]).drop_duplicates(
        ["stock", "headline"], keep="first"
    )
    fallback = processed[["stock", "title"]].merge(
        lookup, left_on=["stock", "title"], right_on=["stock", "headline"], how="left"
    )
    out = out.fillna(fallback["url"])
    return out

 ## 2.4 Load cached ticker history from CSV

In [ ]:
def load_history_from_cache(cache_file: str) -> pd.DataFrame:
    """Read a single per-ticker CSV cache file and return a DatetimeIndex DataFrame."""
    if not os.path.exists(cache_file):
        return pd.DataFrame(columns=FIELDS)
    df = pd.read_csv(cache_file)
    if df.empty or "Date" not in df.columns:
        return pd.DataFrame(columns=FIELDS)

    idx = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    out = pd.DataFrame(index=idx)
    for f in FIELDS:
        out[f] = pd.to_numeric(df[f], errors="coerce").to_numpy() if f in df.columns else np.nan
    out = out[~out.index.isna()]
    out = out[~out.index.duplicated(keep="last")]
    out = out.sort_index()
    return out

## 2.5 Compute OHLCV Feature Windows (t-10 ... t+1)

In [ ]:
def compute_features_from_local(
    processed: pd.DataFrame,
    status_df: pd.DataFrame,
    drop_incomplete_rows: bool = True,
) -> pd.DataFrame:
    """
    For each news row, look up the stock's cached OHLCV and fill in:
      - t_minus_1 ... t_minus_10  (previous 10 trading days)
      - t_plus_1                  (next trading day)
    Returns the processed DataFrame with 55 new feature columns appended.
    """
    feature_cols = make_feature_columns()
    n = len(processed)
    feat = np.full((n, len(feature_cols)), np.nan, dtype=np.float64)

    minus_col = {(i, fi): (i - 1) * len(FIELDS) + fi for i in range(1, 11) for fi in range(len(FIELDS))}
    plus_col  = {fi: 10 * len(FIELDS) + fi for fi in range(len(FIELDS))}

    stock_to_file = (
        status_df.drop_duplicates(["stock"], keep="last")[["stock", "cache_file", "status"]]
        .set_index("stock").to_dict("index")
    )

    keep_mask = np.zeros(n, dtype=bool)
    news_days_all = processed["news_day"].values.astype("datetime64[D]")

    grouped = processed.groupby("stock", sort=False).indices
    total = len(grouped)

    for gi, (stock, idxs) in enumerate(grouped.items(), start=1):
        meta = stock_to_file.get(stock)
        if meta is None or meta.get("status") in {"empty", "rate_limited", "bad_cache"}:
            if gi % 200 == 0 or gi == total:
                print(f"Feature progress: {gi}/{total} stocks", flush=True)
            continue

        hist = load_history_from_cache(meta["cache_file"])
        if hist.empty:
            continue

        idxs_arr  = np.asarray(idxs)
        stock_days = news_days_all[idxs_arr]
        valid_day  = ~np.isnat(stock_days)
        if not np.any(valid_day):
            continue

        row_idxs  = idxs_arr[valid_day]
        news_days = stock_days[valid_day]
        dates   = hist.index.values.astype("datetime64[D]")
        arrays  = [hist[f].to_numpy(dtype=np.float64, copy=False) for f in FIELDS]
        left    = np.searchsorted(dates, news_days, side="left")

        # Previous 10 trading days
        for i in range(1, 11):
            prev_idx = left - i
            ok = prev_idx >= 0
            if not np.any(ok):
                continue
            for fi, arr in enumerate(arrays):
                feat[row_idxs[ok], minus_col[(i, fi)]] = arr[prev_idx[ok]]

        # Next trading day (t+1)
        inb = left < len(dates)
        eq  = np.zeros_like(inb, dtype=bool)
        eq[inb] = dates[left[inb]] == news_days[inb]
        nxt = left + eq.astype(np.int64)
        okn = nxt < len(dates)
        if np.any(okn):
            for fi, arr in enumerate(arrays):
                feat[row_idxs[okn], plus_col[fi]] = arr[nxt[okn]]

        keep_mask[row_idxs] = True

        if gi % 200 == 0 or gi == total:
            print(f"Feature progress: {gi}/{total} stocks", flush=True)

    out = processed.copy()
    out = pd.concat([out, pd.DataFrame(feat, columns=feature_cols)], axis=1)
    out = out[keep_mask]
    if drop_incomplete_rows:
        out = out.dropna(subset=feature_cols)
    out = out.reset_index(drop=True)
    return out

## 2.6: Fetch Quote Type

In [ ]:
def fetch_quote_types(tickers, cache_path):
    """Get yfinance quoteType for each ticker. Uses cache CSV if it exists."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        return pd.read_csv(cache_path)

    records = []
    for i, t in enumerate(tickers):
        try:
            info = yf.Ticker(t).info
            qt = info.get("quoteType", "UNKNOWN")
        except Exception:
            qt = "ERROR"
        records.append({"symbol": t, "quoteType": qt})
        if (i + 1) % 100 == 0:
            print(f"quoteType progress: {i+1}/{len(tickers)}", flush=True)
    result = pd.DataFrame(records)
    result.to_csv(cache_path, index=False)
    return result

## 2.7 Fetch Sector and Industry

In [ ]:
def fetch_ticker_metadata(tickers, cache_path):
    """Get sector & industry for each ticker via yfinance. Uses cache CSV if it exists."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        return pd.read_csv(cache_path)

    records = []
    for i, t in enumerate(tickers):
        try:
            info = yf.Ticker(t).info
            records.append({"symbol": t, "sector": info.get("sector"), "industry": info.get("industry")})
        except Exception:
            records.append({"symbol": t, "sector": None, "industry": None})
        if (i + 1) % 100 == 0:
            print(f"metadata progress: {i+1}/{len(tickers)}", flush=True)
    result = pd.DataFrame(records)
    result.to_csv(cache_path, index=False)
    return result

## 2.8 Fetch Earning Report

In [ ]:
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

def fetch_earnings_all(tickers, cache_path):
    """Fetch earnings dates for all tickers via yf.Ticker().get_earnings_dates(). Uses cache."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        return pd.read_csv(cache_path)

    all_rows = []
    for i, t in enumerate(tickers):
        try:
            ed = yf.Ticker(t).get_earnings_dates(limit=60)
            if ed is not None and not ed.empty:
                ed = ed.reset_index()
                ed["symbol"] = t
                ed = ed.rename(columns={"Earnings Date": "earnings_date"})
                all_rows.append(ed)
        except Exception:
            pass
        if (i + 1) % 100 == 0:
            print(f"earnings progress: {i+1}/{len(tickers)}", flush=True)

    result = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    result.to_csv(cache_path, index=False)
    return result

# Part 3: Data Wrangling and Cleaning

## 3.1 Load data and url

In [ ]:
# Load the two source CSVs
processed = pd.read_csv(ORIG_DIR / "analyst_ratings_processed.csv",
                        usecols=["Unnamed: 0", "title", "date", "stock"])
raw = pd.read_csv(ORIG_DIR / "raw_analyst_ratings.csv",
                  usecols=["Unnamed: 0", "headline", "url", "stock"])

print(f"Processed rows: {len(processed):,}")
print(f"Raw rows:       {len(raw):,}")

Processed rows: 1,400,469
Raw rows:       1,407,328


In [ ]:
# Merge URLs from raw onto processed
processed["url"] = merge_url(processed, raw)
print(f"URL coverage: {processed['url'].notna().mean():.2%}")


URL coverage: 93.31%


In [ ]:
# Parse news timestamps -> calendar date (ET)
news_ts = pd.to_datetime(processed["date"], utc=True, errors="coerce")
processed["news_day"] = (
    news_ts.dt.tz_convert("America/New_York")
          .dt.tz_localize(None)
          .dt.normalize()
)
print(f"Date range: {processed['news_day'].min()} to {processed['news_day'].max()}")

Date range: 2009-02-14 00:00:00 to 2020-06-11 00:00:00


## 3.2 Preprocess OHLCV files

In [ ]:
# Build a status DataFrame from the pre-existing OHLCV cache directory.
# This replaces the download step — we just scan what's already on disk.

cache_dir = str(OHLCV_DIR)
stocks = sorted(processed["stock"].dropna().astype(str).unique().tolist())

rows = []
for s in stocks:
    q = normalize_query_ticker(s)
    if not q:
        continue
    fname = safe_filename(q) + ".csv"
    fpath = os.path.join(cache_dir, fname)
    if os.path.exists(fpath):
        rows.append({"stock": s, "query_ticker": q, "cache_file": fpath, "status": "cached", "rows": -1, "message": ""})
    else:
        rows.append({"stock": s, "query_ticker": q, "cache_file": fpath, "status": "missing", "rows": 0, "message": "no_cache_file"})

status_df = pd.DataFrame(rows)
print(f"Total stocks: {len(status_df)}")
print(status_df["status"].value_counts())

Total stocks: 6192
status
cached    6192
Name: count, dtype: int64


## 3.3 Compute features for OHLCV and obtain expanded dataset

In [ ]:
# Compute the 55 OHLCV feature columns from cached data
expanded = compute_features_from_local(
    processed=processed,
    status_df=status_df,
    drop_incomplete_rows=True,
)

# Keep only the final output columns
final_cols = ["title", "date", "stock", "url"] + make_feature_columns()
expanded = expanded[final_cols]
OUTPUT_PATH = DATA_DIR / "analyst_ratings_expanded_local.csv"
expanded.to_csv(OUTPUT_PATH, index=False)

print(f"Output rows : {len(expanded):,}")
print(f"Output cols : {len(expanded.columns)}")
print(f"URL coverage: {expanded['url'].notna().mean():.2%}")

Feature progress: 200/6192 stocks
Feature progress: 400/6192 stocks
Feature progress: 600/6192 stocks
Feature progress: 800/6192 stocks
Feature progress: 1000/6192 stocks
Feature progress: 1200/6192 stocks
Feature progress: 1400/6192 stocks
Feature progress: 1600/6192 stocks
Feature progress: 1800/6192 stocks
Feature progress: 2000/6192 stocks
Feature progress: 2200/6192 stocks
Feature progress: 2400/6192 stocks
Feature progress: 2600/6192 stocks
Feature progress: 2800/6192 stocks
Feature progress: 3000/6192 stocks
Feature progress: 3200/6192 stocks
Feature progress: 3400/6192 stocks
Feature progress: 3600/6192 stocks
Feature progress: 3800/6192 stocks
Feature progress: 4000/6192 stocks
Feature progress: 4200/6192 stocks
Feature progress: 4400/6192 stocks
Feature progress: 4600/6192 stocks
Feature progress: 4800/6192 stocks
Feature progress: 5000/6192 stocks
Feature progress: 5200/6192 stocks
Feature progress: 5400/6192 stocks
Feature progress: 5600/6192 stocks
Feature progress: 5800/6

In [ ]:
expanded.head()

,title,date,stock,url,t_minus_1_open,t_minus_1_high,t_minus_1_low,t_minus_1_close,t_minus_1_volume,t_minus_2_open,...,t_minus_10_open,t_minus_10_high,t_minus_10_low,t_minus_10_close,t_minus_10_volume,t_plus_1_open,t_plus_1_high,t_plus_1_low,t_plus_1_close,t_plus_1_volume
0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A,https://www.benzinga.com/news/20/06/16190091/s...,89.820000,91.739998,89.820000,91.139999,2227500.0,90.650002,...,81.720001,82.190002,80.459999,80.750000,2576500.0,89.309998,90.589996,89.059998,90.290001,1804700.0
1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A,https://www.benzinga.com/news/20/06/16170189/s...,90.000000,90.629997,89.110001,90.290001,1682800.0,88.040001,...,83.230003,83.540001,81.889999,81.970001,2099200.0,89.820000,91.739998,89.820000,91.139999,2227500.0
2,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A,https://www.benzinga.com/news/20/05/16103463/7...,85.000000,87.669998,84.199997,84.980003,5063100.0,81.720001,...,79.599998,81.989998,79.500000,81.269997,1691600.0,86.300003,86.480003,84.370003,86.180000,1917600.0
3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A,https://www.benzinga.com/news/20/05/16095921/4...,81.720001,82.190002,80.459999,80.750000,2576500.0,82.980003,...,79.669998,80.330002,79.190002,79.709999,2255000.0,86.230003,86.790001,85.639999,86.129997,3173400.0
4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A,https://www.benzinga.com/news/20/05/16095304/b...,81.720001,82.190002,80.459999,80.750000,2576500.0,82.980003,...,79.669998,80.330002,79.190002,79.709999,2255000.0,86.230003,86.790001,85.639999,86.129997,3173400.0


## 3.4: Filter equity only

In [ ]:
tickers = expanded.reset_index()["stock"].unique().tolist() if "stock" not in expanded.columns else expanded["stock"].unique().tolist()

qt = fetch_quote_types(tickers, DATA_DIR / "ticker_quotetype.csv")
print(qt["quoteType"].value_counts())

tickers = expanded.reset_index()["stock"].unique().tolist() if "stock" not in expanded.columns else expanded["stock"].unique().tolist()
qt = fetch_quote_types(tickers, DATA_DIR / "ticker_quotetype.csv")
print(qt["quoteType"].value_counts())

# Keep only EQUITY rows
equity_tickers = qt.loc[qt["quoteType"] == "EQUITY", "symbol"].tolist()

if "stock" not in expanded.columns:
    expanded = expanded.reset_index()

df_eq = expanded[expanded["stock"].isin(equity_tickers)].reset_index(drop=True)
print(f"Rows before: {len(expanded):,}  after: {len(df_eq):,}  dropped: {len(expanded) - len(df_eq):,}")
print(f"Equity tickers: {df_eq['stock'].nunique():,}")

quoteType progress: 100/3580
quoteType progress: 200/3580
quoteType progress: 300/3580
quoteType progress: 400/3580
quoteType progress: 500/3580
quoteType progress: 600/3580
quoteType progress: 700/3580
quoteType progress: 800/3580
quoteType progress: 900/3580
quoteType progress: 1000/3580
quoteType progress: 1100/3580
quoteType progress: 1200/3580
quoteType progress: 1300/3580
quoteType progress: 1400/3580
quoteType progress: 1500/3580
quoteType progress: 1600/3580
quoteType progress: 1700/3580
quoteType progress: 1800/3580
quoteType progress: 1900/3580
quoteType progress: 2000/3580
quoteType progress: 2100/3580
quoteType progress: 2200/3580
quoteType progress: 2300/3580
quoteType progress: 2400/3580
quoteType progress: 2500/3580
quoteType progress: 2600/3580
quoteType progress: 2700/3580
quoteType progress: 2800/3580
quoteType progress: 2900/3580
quoteType progress: 3000/3580
quoteType progress: 3100/3580
quoteType progress: 3200/3580
quoteType progress: 3300/3580
quoteType progress:

## 3.5 Fetch Sector & Industry Metadata

In [ ]:
meta = fetch_ticker_metadata(equity_tickers, DATA_DIR / "ticker_metadata.csv")
print(f"Shape: {meta.shape}")
print(f"\nNull counts:\n{meta.isnull().sum()}")
print(f"\nSector distribution:\n{meta['sector'].value_counts()}")

# Join metadata onto equities dataset & drop rows missing sector/industry
df_eq_meta = df_eq.merge(meta, left_on="stock", right_on="symbol", how="left").drop(columns="symbol")
df_eq_meta = df_eq_meta.dropna(subset=["sector", "industry"]).reset_index(drop=True)

print(f"Final shape: {df_eq_meta.shape}")
nulls = df_eq_meta.isnull().sum()
print(f"\nNull counts (non-zero only):\n{nulls[nulls > 0]}")

# Save enriched equities dataset
df_eq_meta.to_csv(DATA_DIR / "expanded_equities.csv", index=False)
df_eq_meta.to_parquet(DATA_DIR / "expanded_equities.parquet", index=False)
print("Saved expanded_equities.csv and .parquet")

metadata progress: 100/865
metadata progress: 200/865


Exception ignored from cffi callback <function buffer_callback at 0x7d5b118a9c60>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/curl_cffi/curl.py", line 101, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 


metadata progress: 300/865


KeyboardInterrupt: 

In [ ]:
df_eq.head()

,title,date,stock,url,t_minus_1_open,t_minus_1_high,t_minus_1_low,t_minus_1_close,t_minus_1_volume,t_minus_2_open,...,t_minus_10_open,t_minus_10_high,t_minus_10_low,t_minus_10_close,t_minus_10_volume,t_plus_1_open,t_plus_1_high,t_plus_1_low,t_plus_1_close,t_plus_1_volume
0,Alphatec Holdings Q1 EPS $(0.330) May Not Comp...,2020-05-11 16:22:00-04:00,ATEC,https://www.benzinga.com/news/earnings/20/05/1...,4.63,4.98,4.63,4.88,329600.0,4.40,...,3.87,4.20,3.80,3.94,767600.0,4.64,4.85,4.33,4.49,699100.0
1,Alphatec Terminates Agreement to Acquire EOS I...,2020-04-27 05:39:00-04:00,ATEC,https://www.benzinga.com/m-a/20/04/15881955/al...,3.81,3.85,3.66,3.83,344900.0,3.79,...,3.98,4.01,3.60,3.93,488900.0,4.11,4.69,4.03,4.54,990000.0
2,H.C. Wainwright Reiterates Buy on Alphatec Hol...,2020-04-09 07:17:00-04:00,ATEC,https://www.benzinga.com/news/20/04/15773750/h...,3.64,4.11,3.57,4.04,829500.0,3.74,...,3.16,3.72,3.08,3.70,1289200.0,3.98,4.01,3.60,3.93,488900.0
3,"ATEC Prelim. Adj. Cash Balance As Of Mar. 31, ...",2020-04-08 09:21:00-04:00,ATEC,https://www.benzinga.com/news/20/04/15766622/a...,3.74,3.74,3.40,3.57,718700.0,3.40,...,2.87,3.25,2.71,3.05,1502100.0,4.18,4.18,3.78,3.94,656700.0
4,ATEC Highlights Draw Down Of $20M Credit Facil...,2020-04-08 09:21:00-04:00,ATEC,https://www.benzinga.com/news/20/04/15766613/a...,3.74,3.74,3.40,3.57,718700.0,3.40,...,2.87,3.25,2.71,3.05,1502100.0,4.18,4.18,3.78,3.94,656700.0


In [ ]:
expanded

,title,date,stock,url,t_minus_1_open,t_minus_1_high,t_minus_1_low,t_minus_1_close,t_minus_1_volume,t_minus_2_open,...,t_minus_10_open,t_minus_10_high,t_minus_10_low,t_minus_10_close,t_minus_10_volume,t_plus_1_open,t_plus_1_high,t_plus_1_low,t_plus_1_close,t_plus_1_volume
0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 10:30:00-04:00,A,https://www.benzinga.com/news/20/06/16190091/s...,89.820000,91.739998,89.820000,91.139999,2227500.0,90.650002,...,81.720001,82.190002,80.459999,80.750000,2576500.0,89.309998,90.589996,89.059998,90.290001,1804700.0
1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 10:45:00-04:00,A,https://www.benzinga.com/news/20/06/16170189/s...,90.000000,90.629997,89.110001,90.290001,1682800.0,88.040001,...,83.230003,83.540001,81.889999,81.970001,2099200.0,89.820000,91.739998,89.820000,91.139999,2227500.0
2,71 Biggest Movers From Friday,2020-05-26 04:30:00-04:00,A,https://www.benzinga.com/news/20/05/16103463/7...,85.000000,87.669998,84.199997,84.980003,5063100.0,81.720001,...,79.599998,81.989998,79.500000,81.269997,1691600.0,86.300003,86.480003,84.370003,86.180000,1917600.0
3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 12:45:00-04:00,A,https://www.benzinga.com/news/20/05/16095921/4...,81.720001,82.190002,80.459999,80.750000,2576500.0,82.980003,...,79.669998,80.330002,79.190002,79.709999,2255000.0,86.230003,86.790001,85.639999,86.129997,3173400.0
4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 11:38:00-04:00,A,https://www.benzinga.com/news/20/05/16095304/b...,81.720001,82.190002,80.459999,80.750000,2576500.0,82.980003,...,79.669998,80.330002,79.190002,79.709999,2255000.0,86.230003,86.790001,85.639999,86.129997,3173400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
849704,"Zumiez Beats, Reverses Loss - Analyst Blog",2010-08-23 08:20:00-04:00,ZUMZ,https://www.benzinga.com/10/08/438769/zumiez-b...,16.450001,16.469999,15.180000,15.740000,375000.0,16.459999,...,17.190001,17.270000,16.930000,17.150000,319300.0,15.780000,15.780000,14.940000,14.970000,338200.0
849705,Jefferies Reiterates Hold Rating on Zumiez (ZUMZ),2010-08-20 09:58:00-04:00,ZUMZ,https://www.benzinga.com/analyst-ratings/analy...,16.459999,16.910000,15.770000,15.950000,334200.0,16.250000,...,17.290001,17.610001,16.900000,16.969999,251400.0,15.900000,16.629999,15.660000,15.910000,253300.0
849706,Zumiez Beats Earnings Estimates (ZUMZ),2010-08-19 16:13:00-04:00,ZUMZ,https://www.benzinga.com/markets/company-news/...,16.250000,16.709999,16.150000,16.480000,361300.0,16.750000,...,18.270000,18.370001,17.180000,17.360001,321300.0,16.450001,16.469999,15.180000,15.740000,375000.0
849707,Piper Jaffray Overweight On Zumiez (ZUMZ),2010-08-06 12:42:00-04:00,ZUMZ,https://www.benzinga.com/markets/company-news/...,18.270000,18.370001,17.180000,17.360001,321300.0,17.900000,...,18.680000,19.389999,18.250000,19.290001,158800.0,17.190001,17.270000,16.930000,17.150000,319300.0


# Part 4: Earning report (Further study)

In [ ]:
earn = fetch_earnings_all(equity_tickers, DATA_DIR / "earnings_dates.csv")
print(f"Earnings records: {len(earn):,}")
earn.head(5)

ERROR:yfinance:AVK: No earnings dates found, symbol may be delisted
ERROR:yfinance:AWF: No earnings dates found, symbol may be delisted
ERROR:yfinance:AWP: No earnings dates found, symbol may be delisted
ERROR:yfinance:AWX: No earnings dates found, symbol may be delisted
ERROR:yfinance:BBDO: No earnings dates found, symbol may be delisted
ERROR:yfinance:BCV: No earnings dates found, symbol may be delisted
ERROR:yfinance:BDL: No earnings dates found, symbol may be delisted
ERROR:yfinance:BFK: No earnings dates found, symbol may be delisted
ERROR:yfinance:BFZ: No earnings dates found, symbol may be delisted
ERROR:yfinance:BGI: No earnings dates found, symbol may be delisted
ERROR:yfinance:BGY: No earnings dates found, symbol may be delisted
ERROR:yfinance:BHK: No earnings dates found, symbol may be delisted
ERROR:yfinance:BIT: No earnings dates found, symbol may be delisted
ERROR:yfinance:BKN: No earnings dates found, symbol may be delisted
ERROR:yfinance:BKT: No earnings dates found, sy

earnings progress: 100/419


ERROR:yfinance:BLW: No earnings dates found, symbol may be delisted
ERROR:yfinance:BME: No earnings dates found, symbol may be delisted
ERROR:yfinance:BMS: No earnings dates found, symbol may be delisted
ERROR:yfinance:BNY: No earnings dates found, symbol may be delisted
ERROR:yfinance:BPOPM: No earnings dates found, symbol may be delisted
ERROR:yfinance:BSL: No earnings dates found, symbol may be delisted
ERROR:yfinance:BST: No earnings dates found, symbol may be delisted
ERROR:yfinance:BTA: No earnings dates found, symbol may be delisted
ERROR:yfinance:BTT: No earnings dates found, symbol may be delisted
ERROR:yfinance:BWG: No earnings dates found, symbol may be delisted
ERROR:yfinance:BYFC: No earnings dates found, symbol may be delisted
ERROR:yfinance:CAF: No earnings dates found, symbol may be delisted


earnings progress: 200/419


ERROR:yfinance:CEV: No earnings dates found, symbol may be delisted
ERROR:yfinance:CGO: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHEV: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHI: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHNR: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHSCM: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHSCN: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHSCO: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHSCP: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHW: No earnings dates found, symbol may be delisted
ERROR:yfinance:CHY: No earnings dates found, symbol may be delisted
ERROR:yfinance:CIF: No earnings dates found, symbol may be delisted
ERROR:yfinance:CII: No earnings dates found, symbol may be delisted
ERROR:yfinance:CIK: No earnings dates found, symbol may be delisted
ERROR:yfinance:CKX: No earnings dates 

...

ERROR:yfinance:CRF: No earnings dates found, symbol may be delisted
ERROR:yfinance:CRT: No earnings dates found, symbol may be delisted
ERROR:yfinance:CSQ: No earnings dates found, symbol may be delisted
ERROR:yfinance:CXE: No earnings dates found, symbol may be delisted
ERROR:yfinance:CXH: No earnings dates found, symbol may be delisted
ERROR:yfinance:DBL: No earnings dates found, symbol may be delisted
ERROR:yfinance:DDT: No earnings dates found, symbol may be delisted
ERROR:yfinance:DFP: No earnings dates found, symbol may be delisted


earnings progress: 400/419


ERROR:yfinance:DIAX: No earnings dates found, symbol may be delisted
ERROR:yfinance:EXG: No earnings dates found, symbol may be delisted
ERROR:yfinance:GAM: No earnings dates found, symbol may be delisted
ERROR:yfinance:HOVNP: No earnings dates found, symbol may be delisted
ERROR:yfinance:KIO: No earnings dates found, symbol may be delisted
ERROR:yfinance:MFM: No earnings dates found, symbol may be delisted
ERROR:yfinance:TAIT: No earnings dates found, symbol may be delisted
ERROR:yfinance:TCCO: No earnings dates found, symbol may be delisted


Earnings records: 26,141


,earnings_date,EPS Estimate,Reported EPS,Surprise(%),symbol
0,2026-04-30 16:00:00-04:00,NaN,NaN,NaN,ATEC
1,2026-02-24 16:00:00-05:00,-0.14,-0.14,-2.94,ATEC
2,2025-10-30 16:00:00-04:00,-0.16,-0.19,-16.33,ATEC
3,2025-07-31 16:00:00-04:00,-0.19,-0.27,-43.37,ATEC
4,2025-05-01 16:00:00-04:00,-0.24,-0.35,-48.49,ATEC


In [ ]:
# Build stock-centric OHLCV dataset (deduplicated by stock+date)
ohlcv_cols = [c for c in df_eq_meta.columns if c.startswith("t_")]
keep_cols  = ["stock", "date"] + ohlcv_cols + ["sector", "industry"]

stock_ohlcv = (
    df_eq_meta[keep_cols]
    .assign(date=pd.to_datetime(df_eq_meta["date"], utc=True).dt.normalize().dt.tz_localize(None))
    .drop_duplicates(subset=["stock", "date"])
    .sort_values(["stock", "date"])
    .reset_index(drop=True)
)
print(f"Deduplicated stock-OHLCV rows: {stock_ohlcv.shape}")

Deduplicated stock-OHLCV rows: (79781, 59)


In [ ]:
# Normalize earnings date for join
earn_join = earn.copy()
earn_join["date"] = (
    pd.to_datetime(earn_join["earnings_date"], utc=True)
      .dt.normalize()
      .dt.tz_localize(None)
)
earn_join = earn_join.drop(columns="earnings_date")

# Check overlap
earn_keys  = set(zip(earn_join["symbol"], earn_join["date"]))
ohlcv_keys = set(zip(stock_ohlcv["stock"], stock_ohlcv["date"]))
matched    = earn_keys & ohlcv_keys

print(f"Earnings records       : {len(earn_join):,}")
print(f"Matched to OHLCV rows  : {len(matched):,}")
print(f"Match rate             : {len(matched)/len(earn_keys)*100:.1f}%")

Earnings records       : 26,141
Matched to OHLCV rows  : 8,416
Match rate             : 32.3%


In [ ]:
# Inner join earnings onto stock_ohlcv
earning_final = stock_ohlcv.merge(
    earn_join.rename(columns={"symbol": "stock"}),
    on=["stock", "date"],
    how="inner",
)

print(f"Final shape: {earning_final.shape}")
earning_final.head()

Final shape: (8422, 62)


,stock,date,t_minus_1_open,t_minus_1_high,t_minus_1_low,t_minus_1_close,t_minus_1_volume,t_minus_2_open,t_minus_2_high,t_minus_2_low,...,t_plus_1_open,t_plus_1_high,t_plus_1_low,t_plus_1_close,t_plus_1_volume,sector,industry,EPS Estimate,Reported EPS,Surprise(%)
0,ATEC,2012-02-28,24.000000,24.240000,23.400000,23.520000,5500.0,23.280001,24.840000,23.160000,...,24.000000,24.000000,21.840000,22.559999,26592.0,Healthcare,Medical Devices,-0.03,-0.72,-2300.00
1,ATEC,2013-04-30,21.360001,23.040001,21.360001,22.200001,14750.0,21.840000,21.959999,21.360001,...,22.799999,23.280001,21.360001,21.480000,28892.0,Healthcare,Medical Devices,-0.04,-0.18,-347.45
2,ATEC,2013-08-06,27.360001,27.600000,27.000000,27.480000,4733.0,27.480000,27.959999,27.120001,...,26.760000,27.000000,24.360001,26.400000,17858.0,Healthcare,Medical Devices,-0.36,-0.12,66.67
3,ATEC,2013-11-07,23.280001,23.280001,22.320000,22.320000,3325.0,23.040001,23.160000,22.680000,...,21.959999,22.440001,21.120001,22.200001,9692.0,Healthcare,Medical Devices,-0.20,-0.24,-21.20
4,ATEC,2014-03-17,16.799999,17.400000,16.559999,16.680000,95367.0,16.920000,16.920000,16.320000,...,21.120001,21.480000,19.799999,20.400000,183725.0,Healthcare,Medical Devices,-0.18,-0.24,-33.33


In [ ]:
# Save final earnings-joined dataset
earning_final.to_parquet(DATA_DIR / "earnings_ONLY_joint.parquet", index=False)
earning_final.to_csv(DATA_DIR / "earnings_ONLY_joint.csv", index=False)
print("Saved earnings_ONLY_joint.parquet and .csv")
print(f"Final rows: {len(earning_final):,}")

Saved earnings_ONLY_joint.parquet and .csv
Final rows: 8,422
